# Detección de desviaciones
Proceso para identificar cambios que disminuyen el desempeño del modelo con el tiempo. Esto es crucial para mantener la exactitud y confianza del modelo, para realizar intervenciones a tiempo como reentrenamiento del modelo o adaptación a nueva distribución de datos.
Para simular nuevos datos usamos [_dbldatagen_ library](https://github.com/databrickslabs/dbldatagen).


## 1. Configuracion

In [0]:
%pip install --quiet databricks-sdk mlflow-skinny --upgrade dbldatagen
%restart_python

In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.dropdown("perf_metric", "f1_score.macro", ["accuracy_score", "precision.weighted", "recall.weighted", "f1_score.macro"])
dbutils.widgets.dropdown("drift_metric", "js_distance", ["chi_squared_test.statistic", "chi_squared_test.pvalue", "tv_distance", "l_infinity_distance", "js_distance"])
dbutils.widgets.text("model_id", "*", "Model Id")

In [0]:
metric = dbutils.widgets.get("perf_metric")
drift = dbutils.widgets.get("drift_metric")
model_id = dbutils.widgets.get("model_id")
model_full_name = dbutils.widgets.get("model_full_name")
project_path = dbutils.widgets.get("project_path")
catalog = model_full_name.split('.')[0]
schema = model_full_name.split('.')[1]
model_name = model_full_name.split('.')[-1]

table_inference = "Churn_Inference"
target_col = "churn"

## 2. Refrescar el monitor
Refrescar monitor solo es necesario si la tabla monitor ha sufrido cambios.

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInfoStatus, MonitorRefreshInfoState

w = WorkspaceClient()
refresh_info = w.quality_monitors.run_refresh(table_name=f"{catalog}.{schema}.{table_inference}")

while refresh_info.state in (MonitorRefreshInfoState.PENDING, MonitorRefreshInfoState.RUNNING):
  refresh_info = w.quality_monitors.get_refresh(table_name=f"{catalog}.{schema}.{table_inference}", refresh_id=refresh_info.refresh_id)
  time.sleep(30)

In [0]:
monitor_info = w.quality_monitors.get(table_name=f"{catalog}.{schema}.{table_inference}")
drift_table_name = monitor_info.drift_metrics_table_name
profile_table_name = monitor_info.profile_metrics_table_name

## 3. Inspeccionar dashboard
El dashboard muestra metricas del último desempeño del modelo.

### Recuperar metricas de desviación
Validamos si las métricas exceden ciertos umbrales definidos para el negocio, por ejemplo:
1. Prediction drift (Jensen–Shannon distance) > 0.2
2. Label drift (Jensen–Shannon distance) > 0.2
3. Expected Loss (daily average per user) > 30
4. Performance(i.e. F1-Score) < 0.4

In [0]:
performance_metrics_df = spark.sql(f"""
SELECT
  window.start as time,
  {metric} AS performance_metric,
  expected_loss,
  Model_Version AS `Model Id`
FROM {profile_table_name}
WHERE
  window.start >= "2025-06-28"
	AND log_type = "INPUT"
  AND column_name = ":table"
  AND slice_key is null
  AND slice_value is null
  AND Model_Version = '{model_id}'
ORDER BY
  window.start
"""
)
display(performance_metrics_df)

In [0]:
drift_metrics_df = spark.sql(f"""
  SELECT
  window.start AS time,
  column_name,
  {drift} AS drift_metric,
  Model_Version AS `Model Id`
FROM {drift_table_name}
WHERE
  column_name IN ('prediction', '{target_col}')
  AND window.start >= "2025-06-30"
  AND slice_key is null
  AND slice_value is null
  AND Model_Version = '{model_id}'
  AND drift_type = "CONSECUTIVE"
ORDER BY
  window.start
"""
)
display(drift_metrics_df )

In [0]:
from pyspark.sql.functions import first

# If no drift on the label or prediction, we skip it
if not drift_metrics_df.isEmpty():
    unstacked_drift_metrics_df = (
        drift_metrics_df.groupBy("time", "`Model Id`")
        .pivot("column_name")
        .agg(first("drift_metric"))
        .orderBy("time")
    )
    display(unstacked_drift_metrics_df)

In [0]:
all_metrics_df = performance_metrics_df
if not drift_metrics_df.isEmpty():
    all_metrics_df = performance_metrics_df.join(
        unstacked_drift_metrics_df, on=["time", "Model Id"], how="inner"
    )

display(all_metrics_df)

## 4. Contar todas las violaciones y guardar como "valor de tarea"
Aqui definimos la diferencia del umbral para las metricas de interes para cualificar una desviación:
- Performance metric < 0.5 
- Average Expected Loss per customer (our custom metric connected to business) > 30 dollars

In [0]:
from pyspark.sql.functions import col, abs

performance_violation_count = all_metrics_df.where(
    (col("performance_metric") < 0.5) & (abs(col("expected_loss")) > 30)
).count()

drift_violation_count = 0
if not drift_metrics_df.isEmpty():
    drift_violation_count = all_metrics_df.where(
        (col(target_col) > 0.19) & (col("prediction") > 0.19)
    ).count()

all_violations_count = drift_violation_count + performance_violation_count
print(f"Total number of violations: {all_violations_count}")

## Siguiente disparar reentrenamiento del modelo
Después de detectar desviaciones tomamos acciones como:
- Reentrenar el modelo
- Enviar una alerta vía Slack o email

El numero de violaciones es almacenado en un "Valor de tarea" y en el job adicionar una rama con una condición if/else para disparar el entrenamiento del modelo.

In [0]:
dbutils.jobs.taskValues.set(key = 'all_violations_count', value = all_violations_count)